In [1]:
%pip install nest_asyncio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from datasets import load_dataset

# load the dataset in streaming mode
docs = load_dataset(
    "aharley/rvl_cdip",
    split="train",
    streaming=True,
    trust_remote_code=True,
 )

#print one of the documents to test
sample = next(iter(docs))
print(sample.keys())

dict_keys(['image', 'label'])


In [ ]:
#install dependencies
%pip install -Uq "datasets<4" transformers torch faiss-cpu pillow requests tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
# import libraries
import os
import re
from itertools import islice

import faiss
import numpy as np
import requests
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoTokenizer,
    DonutProcessor,
    VisionEncoderDecoderModel,
)
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_DOCS = 10
TOP_K = 4

#labels for the dataset
RVL_LABELS = [
    "letter", "form", "email", "handwritten", "advertisement", "scientific_report",
    "scientific_publication", "specification", "file_folder", "news_article", "budget",
    "invoice", "presentation", "questionnaire", "resume", "memo"
]

#print device being used
print(f"device={DEVICE}")

device=cpu


In [18]:
#split rvl_cdip dataset to avoid using the full thing
rvl_stream = load_dataset("aharley/rvl_cdip", split="train", streaming=True, trust_remote_code=True)
raw_examples = list(islice(rvl_stream, MAX_DOCS))

#print number of documents sampled and the keys in the first document
print(f"sampled_docs={len(raw_examples)}")
print("sample_keys:", raw_examples[0].keys())
print("sample_label:", int(raw_examples[0]["label"]), RVL_LABELS[int(raw_examples[0]["label"])])

sampled_docs=10
sample_keys: dict_keys(['image', 'label'])
sample_label: 11 invoice


In [20]:
#import donut model
DONUT_MODEL_NAME = "naver-clova-ix/donut-base-finetuned-docvqa"
donut_processor = DonutProcessor.from_pretrained(DONUT_MODEL_NAME)
donut_model = VisionEncoderDecoderModel.from_pretrained(DONUT_MODEL_NAME).to(DEVICE)
donut_model.eval()

#run donut on an image and return extracted text
def run_donut(image, question="What text content is visible in this document?"):
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    image = image.convert("RGB")

    task_prompt = f"<s_docvqa><s_question>{question}</s_question><s_answer>"
    decoder_input_ids = donut_processor.tokenizer(
        task_prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(DEVICE)

    pixel_values = donut_processor(image, return_tensors="pt").pixel_values.to(DEVICE)

    with torch.no_grad():
        outputs = donut_model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_new_tokens=96,
            pad_token_id=donut_processor.tokenizer.pad_token_id,
            eos_token_id=donut_processor.tokenizer.eos_token_id,
            bad_words_ids=[[donut_processor.tokenizer.unk_token_id]],
            use_cache=True,
        )

    sequence = donut_processor.batch_decode(outputs, skip_special_tokens=False)[0]
    sequence = sequence.replace(donut_processor.tokenizer.eos_token, "")
    sequence = sequence.replace(donut_processor.tokenizer.pad_token, "")
    sequence = re.sub(r"<.*?>", "", sequence).strip()
    return sequence


#run donut on the sampled documents and combine with label info to create ocr_docs
ocr_docs = []
for item in tqdm(raw_examples, desc="Donut extract"):
    label_id = int(item["label"])
    label_name = RVL_LABELS[label_id]

    donut_text = run_donut(item["image"])
    if not donut_text:
        donut_text = f"No text extracted. Likely document class: {label_name}."

    combined_text = f"document_type: {label_name}\nextracted_text: {donut_text}"

    ocr_docs.append({
        "label_id": label_id,
        "label_name": label_name,
        "text": combined_text,
    })

print(f"ocr_docs={len(ocr_docs)}")
print(ocr_docs[0]["text"][:300])

Loading weights: 100%|██████████| 484/484 [00:00<00:00, 28039.66it/s]
The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Donut extract: 100%|██████████| 10/10 [02:08<00:00, 12.85s/it]

ocr_docs=10
document_type: invoice
extracted_text: What text content is visible in this document? invoice


In [21]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(DEVICE)
bert_model.eval()


def embed_texts(texts, batch_size=16):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Embed"):
        batch = texts[i : i + batch_size]
        encoded = bert_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = bert_model(**encoded).last_hidden_state

        mask = encoded["attention_mask"].unsqueeze(-1)
        summed = (outputs * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        mean_pooled = summed / counts

        normalized = torch.nn.functional.normalize(mean_pooled, p=2, dim=1)
        all_embeddings.append(normalized.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0).astype("float32")


corpus_texts = [d["text"] for d in ocr_docs]
corpus_embeddings = embed_texts(corpus_texts)

index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

print("faiss_docs:", index.ntotal)
print("embedding_dim:", corpus_embeddings.shape[1])

c:\Users\UT2022\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\UT2022\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8391.90it/s]
BertModel LOAD REPORT from

faiss_docs: 10
embedding_dim: 768


In [22]:
def retrieve(query, top_k=TOP_K):
    query_embedding = embed_texts([query])
    scores, indices = index.search(query_embedding, top_k)

    hits = []
    for score, idx in zip(scores[0], indices[0]):
        item = ocr_docs[int(idx)]
        hits.append({
            "score": float(score),
            "label_name": item["label_name"],
            "text": item["text"],
        })

    return hits


def _build_prompt(query, retrieved_docs):
    context_blocks = []
    for i, d in enumerate(retrieved_docs, start=1):
        context_blocks.append(
            f"[Doc {i}] score={d['score']:.3f}\n{d['text'][:700]}"
        )

    context = "\n\n".join(context_blocks)
    return f"""You are a document assistant for RVL-CDIP.
Use ONLY the retrieved context to answer the question.
If the answer is not present, say so clearly.

Question:
{query}

Retrieved Context:
{context}

Answer:"""

In [ ]:
#set to what port you have sglang running on
SGLANG_BASE_URL = os.getenv("SGLANG_BASE_URL", "http://127.0.0.1:30000/v1")
#what model you want to use
SGLANG_MODEL = os.getenv("SGLANG_MODEL", "meta-llama/Llama-3.1-8B-Instruct")

#generate answer with sglang
def generate_with_sglang(prompt, max_tokens=256, temperature=0.2):
    url = f"{SGLANG_BASE_URL}/chat/completions"
    payload = {
        "model": SGLANG_MODEL,
        "messages": [
            {"role": "system", "content": "You are a concise and factual document QA assistant."},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"].strip()

#return answer with retrieved docs as context
def answer_rag(query, top_k=TOP_K):
    retrieved_docs = retrieve(query, top_k=top_k)
    prompt = _build_prompt(query, retrieved_docs)

    try:
        answer = generate_with_sglang(prompt)
        backend = "sglang"
    except Exception as ex:
        answer = (
            "SGLang generation failed."
            "SGLANG_BASE_URL/SGLANG_MODEL.\n"
            f"Error: {type(ex).__name__}: {ex}"
        )
        backend = "retrieval-only"

    return {
        "backend": backend,
        "query": query,
        "answer": answer,
        "retrieved_docs": retrieved_docs,
    }

In [ ]:
#set query here
query = "What evidence suggests a document is an invoice?"
result = answer_rag(query, top_k=4)

#print the results
print("backend:", result["backend"])
print("query:", result["query"])
print("\nanswer:\n", result["answer"])

#print top retrieved documents
print("\nTop retrieved docs:")
for i, d in enumerate(result["retrieved_docs"], start=1):
    print(f"\n[{i}] score={d['score']:.3f} label={d['label_name']}")
    print(d["text"][:250])

Embed: 100%|██████████| 1/1 [00:00<00:00, 25.87it/s]


backend: retrieval-only
query: What evidence suggests a document is an invoice?

answer:
 SGLang generation failed. Start an sglang server and set SGLANG_BASE_URL/SGLANG_MODEL.
Error: ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=30000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=30000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

Top retrieved docs:

[1] score=0.704 label=budget
document_type: budget
extracted_text: What text content is visible in this document? statement aug 19 1988

[2] score=0.699 label=news_article
document_type: news_article
extracted_text: What text content is visible in this document? receipt mail

[3] score=0.697 label=advertisement
document_type: advertisement
extracted_text: What text content is visible in this document? confidential

[4] score=0.696 label=scientific_publication
d